# O-RAN Energy Optimization — Version 2 (Advanced AI/ML + DRL)

This is the **advanced research version**, intended to follow Version 1 in the same manner as the two-version structure of the Massive MIMO repository.

## Main additions
- PyTorch supervised MLP for per-user traffic-demand prediction
- PPO actor-critic for continuous power allocation
- 64-antenna, 8-user uplink channel model
- Explicit MRC multi-user interference in the SINR calculation
- Joint optimization of user power shares and total transmit-power fraction
- Energy efficiency, throughput, QoS satisfaction, outage, fairness and power reporting
- Equal-power, channel-heuristic and SLSQP numerical-optimization comparators

## Mathematical formulation

$
\mathbf y=\mathbf H\mathbf P^{1/2}\mathbf x+\mathbf n,\quad
\mathbf n\sim\mathcal{CN}(0,\sigma^2\mathbf I)
$

For MRC, (\mathbf w_k=\mathbf h_k):
$
\gamma_k=
\frac{p_k|\mathbf h_k^H\mathbf h_k|^2}
{\sum_{i\ne k}p_i|\mathbf h_k^H\mathbf h_i|^2+
\sigma^2\|\mathbf h_k\|^2}.
$
$
R_k=B\log_2(1+\gamma_k),\quad
EE=\frac{\sum_kR_k}{P_{static}+P_{circuit}+\sum_kp_k/\eta_{PA}}.
$

The QoS target is traffic-dependent. The PPO reward combines normalized energy efficiency and throughput, and penalizes QoS outage and abrupt power changes.

**Important:** this is a simulator prototype. It does not implement live A1/E2 interfaces, a production xApp/rApp, ZF/MMSE receivers or calibrated hardware power consumption. PPO is not guaranteed to outperform SLSQP. Use the validation protocol before making claims.

In [ ]:
# Locate and activate the Version 2 Python package.
from pathlib import Path
import os, sys, subprocess

ROOT = Path.cwd()
ADV = ROOT / "experiments" / "AI-Enabled-O-RAN-ML-DRL-QoS-Optimization"
if not ADV.exists():
    raise FileNotFoundError(f"Version 2 package not found: {ADV}")
os.chdir(ADV)
sys.path.insert(0, str(ADV))
print("Version 2 working directory:", ADV)


In [ ]:
# Install the Version 2 dependencies.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Step 1 — Train the supervised traffic predictor

The MLP predicts synthetic per-user traffic demand from recent demand history. The training script reports test MAE against a persistence baseline. The predictor must beat that baseline consistently across independent seeds before claiming predictive value.

In [ ]:
subprocess.run([sys.executable, "-m", "src.train_traffic"], check=True)


## Step 2 — Train PPO

The PPO action selects:
1. The relative power share for each of the eight users.
2. The fraction of the total 10 W transmit-power budget to use.

A softmax maps user logits to nonnegative shares; a sigmoid selects total power fraction. The resulting power vector respects the sum-power budget.

In [ ]:
subprocess.run([sys.executable, "-m", "src.train_ppo"], check=True)


## Step 3 — Evaluate against conventional methods

The evaluation uses identical channel/traffic samples for four methods:
- Equal power
- Channel-aware heuristic
- SLSQP numerical optimizer
- PPO + traffic ML

The evaluation writes block-level observations and mean/standard-deviation summaries. PPO must be compared against these baselines; do not assume it wins.

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.evaluate",
    "--episodes", "5", "--blocks", "100", "--seed", "29"
], check=True)


In [ ]:
import pandas as pd
from pathlib import Path

results = pd.read_csv("results/evaluation.csv")
summary = pd.read_csv("results/summary.csv", header=[0,1], index_col=0)
display(results.head())
display(summary)


In [ ]:
import matplotlib.pyplot as plt

metric = "energy_efficiency_mbps_per_w"
plot_df = results.groupby("method")[metric].mean().sort_values(ascending=False)
ax = plot_df.plot(kind="bar", xlabel="Allocation method", ylabel="Mean energy efficiency (Mbps/W)",
                  title="Version 2 — Energy-efficiency comparison", rot=20)
plt.tight_layout()
plt.show()

metric = "qos_satisfaction"
plot_df = results.groupby("method")[metric].mean().sort_values(ascending=False)
ax = plot_df.plot(kind="bar", xlabel="Allocation method", ylabel="Mean QoS satisfaction",
                  title="Version 2 — QoS comparison", rot=20)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## Reproducibility and research checklist

Before presenting Version 2 as a research contribution:
- Train PPO with the full configured schedule.
- Evaluate at least 5–10 independent seeds and hundreds/thousands of blocks.
- Report confidence intervals, not one run's mean.
- Tune reward weights only on validation seeds and freeze them for testing.
- Run ablations with/without traffic prediction and QoS penalty.
- Compare prediction against persistence.
- State clearly that this is MRC-only, narrowband, synthetic traffic and an assumed power model.

See `VALIDATION_REPORT.md` and the package README for the current prototype limitations.